# Stage 5: global MIEC model

Global fit of sigma(p(O2), T) per process with the three-channel mixed ionic-electronic
model: one parameter set (three sigma0, three Ea) fits the whole surface.

**Reads:** `{sample_id}/Results/{condition}/stage3_fit.xlsx`
**Writes:** `{sample_id}/Results/pO2/stage5_model.xlsx` · `Results/pO2/Stage5_*` figures · `session.json → stage5_params`

**Model:**

$$\sigma = \sum_i \frac{\sigma_{0,i}}{T}\, e^{-E_{a,i}/k_BT}\, p_{\mathrm{O_2}}^{\pm x_i}$$

summed over the ion ($x=0$), p and n channels ($\sigma_{0,i}$ = prefactor, $E_{a,i}$ =
activation energy, $k_B$ = Boltzmann constant). Needs `p(O2)` data (a `pO2` column or
stages 0-1); skips cleanly without it. `MODEL_CHANNELS` is a defect-chemistry decision,
not a fit outcome.

## Quick links

"Step" below numbers the cells in this notebook, not the pipeline Stages 0-5.

- [Configuration](#configuration): MODEL_PEAK_IDS, MODEL_CHANNELS, MODEL_EXPONENT, T / condition window
- [Step 1: Global fit](#step-1-global-fit-figures-and-export): batch fit, figures, export
- [Step 2: Interactive refit](#step-2-interactive-refit): pick conditions / temperatures / channels
- [Output summary](#output-summary): fitted parameters per peak

## Configuration

**Mode switch** `PARAM_MODE` (top of the cell below), hand-edited and the cell re-run to switch:

- `"lock"` = read-only reproduction of the saved calibration, nothing writes back.
- `"continue"` = config cell is the base, starting values load from `session.json` when present, widget/Apply edits merge-save.
- `"reset"` = deliberately ignore `session.json`, start from the literals below, next save overwrites the saved history.

Full semantics: README, "Changing parameters".

In [ ]:
import sys
from pathlib import Path
from pipeline.interactive import select_sample, param_source_banner
from pipeline.session import load_sample, update_sample

NOTEBOOK_DIR = Path.cwd()

sample_id = select_sample(NOTEBOOK_DIR, show_list=True)

_cfg = load_sample(sample_id)

# lock: read-only, no write.
# continue: read session.json if present, edits merge-save.
# reset: ignores session.json, starts from literals below, next save overwrites.
# Full semantics: README, "Changing parameters".
PARAM_MODE = "continue"

_LOCKED_MSG = ("locked (PARAM_MODE='lock'): not saved. "
               "Set PARAM_MODE to 'continue' or 'reset' in Configuration to edit.")

def _update_session(**fields) -> bool:
    """Merge-save to session.json. No-op (returns False) in lock mode."""
    if PARAM_MODE == "lock":
        return False
    update_sample(sample_id, **fields)
    return True

# Starting values load from session.json unless reset: re-run must never
# silently wipe saved tuning.
_p5 = {} if PARAM_MODE == "reset" else _cfg.get("stage5_config", {})
param_source_banner(PARAM_MODE, "Stage 5")

# Processes (Zarc peaks) to fit; [] = every peak found in the data
MODEL_PEAK_IDS = _p5.get("MODEL_PEAK_IDS", [1, 2])

# Brouwer exponent x: 1/4 in the dilute defect regime, 1/6 elsewhere (README.md, MODEL_EXPONENT)
MODEL_EXPONENT = _p5.get("MODEL_EXPONENT", 1/4)

# Conditions (pressures) to include in the fit; [] = all
MODEL_CONDITIONS = _p5.get("MODEL_CONDITIONS", [])

# Temperature window [C] for the fit (None = no limit). The ionic/electronic
# separation is physically reliable only where the peaks separate, e.g.
# MODEL_T_MIN = 475 to drop the lower temperatures where they merge.
MODEL_T_MIN = _p5.get("MODEL_T_MIN", None)
MODEL_T_MAX = _p5.get("MODEL_T_MAX", None)

# Channels of the MIEC model to fit, subset of ["ion", "p", "n"]. This is a
# defect-chemistry decision by the operator, never a fit outcome: e.g. drop
# "n" when the measured pO2 window is never reducing enough to create
# electrons. Excluded channels get sigma0 = 0 and Ea = NaN in every output.
MODEL_CHANNELS = _p5.get("MODEL_CHANNELS", ["ion", "p"])

# Manual (condition, T) validity chosen in stage3 ("Validity selection" cell).
# True = the global fit skips deselected points; xlsx inputs stay complete.
USE_STAGE3_SELECTION = _p5.get("USE_STAGE3_SELECTION", True)
STAGE3_VALID = _cfg.get("stage3_valid", {})

_ = _update_session(stage5_config={
    "MODEL_PEAK_IDS": MODEL_PEAK_IDS, "MODEL_EXPONENT": MODEL_EXPONENT,
    "MODEL_CONDITIONS": MODEL_CONDITIONS,
    "MODEL_T_MIN": MODEL_T_MIN, "MODEL_T_MAX": MODEL_T_MAX,
    "MODEL_CHANNELS": MODEL_CHANNELS,
    "USE_STAGE3_SELECTION": USE_STAGE3_SELECTION,
})

In [ ]:
# inline backend: more reliable than ipympl with ipywidgets panels.
get_ipython().run_line_magic("matplotlib", "inline")  # type: ignore[name-defined]

import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.model import (
    fit_global_conductivity,
    stoichiometric_pO2,
    global_transference_table,
)
from pipeline.plots import (
    apply_pub_style,
    plot_brouwer_transference,
    plot_conductivity_surface_3d,
    plot_fit_residuals,
)
from pipeline.utils import build_metadata_sheet

apply_pub_style()

try:
    import ipywidgets as W
    from IPython.display import display as _display, clear_output as _clear
    _HAS_WIDGETS = True
except Exception as _exc:
    print(f"[INFO] ipywidgets not installed ({_exc}); control panels disabled.")
    _HAS_WIDGETS = False

sample_dir   = NOTEBOOK_DIR / sample_id
RESULTS_BASE = sample_dir / "Results"
print(f"Sample: {sample_id}")

## Step 1: global fit, figures and export

For each selected peak: aggregate sigma(p(O2), T) over the selected conditions and T window,
fit the 6 parameters (VARPRO), and draw the Brouwer + transference figure, the 3-D surface
and the residual map. Activation energies are reported as numbers (printout + Parameters
sheet); no model-Arrhenius plot, since by construction it would be a perfect line.

A structureless residual map and a high global `R2` mean the model describes the data.
Output: `stage5_model.xlsx` (Parameters, Residuals, Metadata) and `session.json → stage5_params`.

In [ ]:
# Aggregate the Peaks rows from every condition that completed Stage 3.
def _load_all_peaks() -> pd.DataFrame:
    frames = []
    if not RESULTS_BASE.exists():
        return pd.DataFrame()
    for d in sorted(RESULTS_BASE.iterdir()):
        f = d / "stage3_fit.xlsx"
        if d.is_dir() and f.exists():
            try:
                df = pd.read_excel(f, sheet_name="Peaks")
            except Exception as exc:
                print(f"[WARN] could not read {f.name}: {type(exc).__name__}: {exc}")
                continue
            if "condition" not in df.columns:
                df["condition"] = d.name
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


df_all_peaks = _load_all_peaks()

if USE_STAGE3_SELECTION and STAGE3_VALID and not df_all_peaks.empty:
    def _row_valid(r) -> bool:
        ts = STAGE3_VALID.get(r["condition"])
        return (ts is None or pd.isna(r["T_nominal"])
                or int(r["T_nominal"]) in {int(t) for t in ts})
    _mask = df_all_peaks.apply(_row_valid, axis=1)
    if (~_mask).any():
        print(f"stage3 selection: dropping {int((~_mask).sum())} "
              f"deselected (condition, T) rows from the fit input")
    df_all_peaks = df_all_peaks[_mask].reset_index(drop=True)

if df_all_peaks.empty:
    print("[SKIP] Stage 5: no stage3_fit.xlsx found. Run Stage 3 for this sample first.")
elif not ("pO2_mean" in df_all_peaks.columns and df_all_peaks["pO2_mean"].notna().any()):
    print("[SKIP] Stage 5: no pO2 data available "
          "(pO2 is recorded only with furnace-log data, stages 0 + 1).")
else:
    if MODEL_CONDITIONS:
        df_all_peaks = df_all_peaks[df_all_peaks["condition"].isin(MODEL_CONDITIONS)]
    peak_ids = (MODEL_PEAK_IDS if MODEL_PEAK_IDS
                else sorted(int(p) for p in df_all_peaks["peak_id"].unique()))
    out_dir = RESULTS_BASE / "pO2"
    param_rows, resid_frames, saved = [], [], {}

    for pid in peak_ids:
        df_peak = df_all_peaks[df_all_peaks["peak_id"] == pid]
        try:
            res = fit_global_conductivity(df_peak, x=MODEL_EXPONENT,
                                          t_min=MODEL_T_MIN, t_max=MODEL_T_MAX,
                                          channels=tuple(MODEL_CHANNELS))
        except ValueError as exc:
            print(f"[SKIP] peak {pid}: {exc}")
            continue
        p, e = res["params"], res["perr"]
        # Report only the channels the operator selected; a selected channel
        # that NNLS drove to zero has no meaningful Ea (leftover of the seed).
        _parts = [f"Peak {pid}: R2={res['r2']:.4f}", f"n={res['n_points']}"]
        for _ch in ("ion", "p", "n"):
            if _ch not in MODEL_CHANNELS:
                continue
            if getattr(p, f"sigma0_{_ch}") == 0.0 or np.isnan(e[f"Ea_{_ch}"]):
                _parts.append(f"Ea_{_ch}: not active (s0 = 0)")
            else:
                _parts.append(f"Ea_{_ch}={getattr(p, f'Ea_{_ch}'):.3f}"
                              f"+/-{e[f'Ea_{_ch}']:.3f} eV")
        print("  ".join(_parts))
        # Conductivity minimum (n=p crossover): meaningful only when BOTH
        # electronic channels are selected and actually present in the fit.
        if {"p", "n"} <= set(MODEL_CHANNELS) and p.sigma0_p > 0 and p.sigma0_n > 0:
            _po2 = df_peak["pO2_mean"].dropna()
            _lo, _hi = (_po2.min(), _po2.max()) if len(_po2) else (None, None)
            for _Tc in sorted(df_peak["T_nominal"].dropna().unique()):
                _pmin = float(stoichiometric_pO2(p, float(_Tc) + 273.15))
                if not np.isfinite(_pmin):
                    continue
                _note = "" if (_lo is not None and _lo <= _pmin <= _hi) else "  (extrapolated)"
                print(f"    conductivity minimum @ {int(_Tc)} C: pO2 = {_pmin:.2e} bar{_note}")
        # Stage-4 Brouwer + transference figure, redrawn from the refined global
        # model (no fake Arrhenius: the Ea values are reported as numbers above
        # and in the Parameters sheet).
        gtab = global_transference_table(df_peak, p, exponent=MODEL_EXPONENT)
        plot_brouwer_transference(df_peak, out_dir, sample_name=sample_id, peak_id=pid,
                                  exponent=MODEL_EXPONENT, df_t=gtab,
                                  params=p, perr=e)
        plot_conductivity_surface_3d(df_peak, p, out_dir, sample_name=sample_id, peak_id=pid)
        plot_fit_residuals(df_peak, p, out_dir, sample_name=sample_id, peak_id=pid)
        plt.show()
        # All 9 parameter columns are kept for schema stability; excluded or
        # inactive channels carry sigma0 = 0.0 and NaN for Ea / Ea_err.
        row = {
            "peak_id": pid, "R2": res["r2"], "n_points": res["n_points"], "x": p.x,
            "sigma0_ion": p.sigma0_ion, "Ea_ion": p.Ea_ion, "Ea_ion_err": e["Ea_ion"],
            "sigma0_p": p.sigma0_p, "Ea_p": p.Ea_p, "Ea_p_err": e["Ea_p"],
            "sigma0_n": p.sigma0_n, "Ea_n": p.Ea_n, "Ea_n_err": e["Ea_n"],
        }
        param_rows.append(row)
        rf = res["residuals"].copy()
        rf.insert(0, "peak_id", pid)
        resid_frames.append(rf)
        saved[str(pid)] = {k: v for k, v in row.items() if k != "peak_id"}

    if param_rows:
        df_params = pd.DataFrame(param_rows)
        df_resid  = pd.concat(resid_frames, ignore_index=True)
        df_meta   = build_metadata_sheet(sample_id, "stage5_model", {
            "MODEL_EXPONENT":   MODEL_EXPONENT,
            "MODEL_CONDITIONS": MODEL_CONDITIONS or "all",
            "MODEL_T_MIN":      MODEL_T_MIN,
            "MODEL_T_MAX":      MODEL_T_MAX,
            "MODEL_PEAK_IDS":   MODEL_PEAK_IDS or "all",
            "MODEL_CHANNELS":   MODEL_CHANNELS,
        })
        out_dir.mkdir(parents=True, exist_ok=True)
        xlsx = out_dir / "stage5_model.xlsx"
        try:
            with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
                df_params.to_excel(w, sheet_name="Parameters", index=False)
                df_resid.to_excel(w,  sheet_name="Residuals",  index=False)
                df_meta.to_excel(w,   sheet_name="Metadata",   index=False)
            print(f"\nSaved: {xlsx.relative_to(NOTEBOOK_DIR)}")
        except Exception as exc:
            print(f"[WARN] could not write {xlsx.name}: {type(exc).__name__}: {exc}")
        if _update_session(stage5_params=saved):
            print("Stage 5 parameters written to session.json")
        else:
            print(_LOCKED_MSG)
    else:
        print("[SKIP] Stage 5: no peak could be fitted with the current selection.")

## Step 2: interactive refit

Pick a peak, restrict conditions / temperatures, choose the channels (a defect-chemistry
decision, e.g. drop `n` when the `p(O2)` window never gets reducing enough), then press **↻ Refit**.

The fit uses the selected points only, and all three figures are redrawn from that same
subset, so what you see is what was fitted. Each replot **overwrites** the Step-1 figures;
`session.json` is updated only on **↻ Refit** (the first automatic fit is a preview).

In [ ]:
# Interactive refit panel: pick a peak, restrict conditions / temperatures /
# channels, press Refit. Light enough to recompute (unlike Stage 4's replot).
# Fits the SELECTED points only, and every figure shows exactly those points.
# All three figures overwrite the Step-1 files; session.json is written only on
# Refit, so merely running this cell cannot overwrite the batch parameters.
#
# Design rule (same as the stage-3/4 panels): NO W.Output. The three plots are
# W.Image widgets and the report a W.HTML, all value-replaced. Rendering each
# figure with fig.savefig (not _display through the inline backend) both avoids
# the double render of pyplot-managed figures AND makes re-fitting update the
# same images in place instead of appending a new set under the old one.
import html as _html
import traceback as _tb
from io import BytesIO as _BytesIO

if (_HAS_WIDGETS and not df_all_peaks.empty
        and "pO2_mean" in df_all_peaks.columns
        and df_all_peaks["pO2_mean"].notna().any()):
    _pids  = sorted(int(p) for p in df_all_peaks["peak_id"].unique())
    _conds = sorted(df_all_peaks["condition"].unique())
    _temps = sorted(int(t) for t in df_all_peaks["T_nominal"].dropna().unique())
    # Start from the configured window, not from every temperature: the batch
    # fit already excluded T outside [MODEL_T_MIN, MODEL_T_MAX] on purpose.
    _t_sel = tuple(t for t in _temps
                   if (MODEL_T_MIN is None or t >= MODEL_T_MIN)
                   and (MODEL_T_MAX is None or t <= MODEL_T_MAX)) or tuple(_temps)

    w_peak  = W.Dropdown(options=_pids, value=_pids[0], description="Peak:",
                         layout=W.Layout(width="180px"))
    w_conds = W.SelectMultiple(options=_conds, value=tuple(_conds), description="Cond:",
                               rows=min(6, len(_conds)), layout=W.Layout(width="440px"))
    w_temps = W.SelectMultiple(options=_temps, value=_t_sel, description="T [°C]:",
                               rows=min(8, len(_temps)), layout=W.Layout(width="180px"))
    # Channel selection is a defect-chemistry decision (see config cell).
    w_chans = {ch: W.Checkbox(value=(ch in MODEL_CHANNELS), description=ch,
                              indent=False, layout=W.Layout(width="70px"))
               for ch in ("ion", "p", "n")}
    w_go    = W.Button(description="↻ Refit", button_style="primary",
                       layout=W.Layout(width="120px"))
    out_txt = W.HTML()                                            # fit report
    img_bt  = W.Image(format="png", layout=W.Layout(width="100%", max_width="1320px"))   # Brouwer transference
    img_s   = W.Image(format="png", layout=W.Layout(width="100%", max_width="880px"))   # sigma(pO2, T) surface
    img_r   = W.Image(format="png", layout=W.Layout(width="100%", max_width="770px"))   # fit residuals
    _busy = [False]

    def _pre(text):
        """Escaped monospace block for a W.HTML value (replaced each refit, never doubles)."""
        return (f"<pre style='margin:0;font:12px/1.4 monospace;white-space:pre-wrap'>"
                f"{_html.escape(text)}</pre>")

    def _to_image(fig, img):
        """Render a (pyplot-managed) figure into a W.Image and release it from pyplot."""
        if fig is None:
            return
        buf = _BytesIO()
        fig.savefig(buf, format="png", dpi=110)
        img.value = buf.getvalue()
        plt.close(fig)

    def _on_refit(_btn=None):
        if _busy[0]:
            return
        _busy[0] = True
        w_go.disabled = True
        try:
            _sel_ch = tuple(ch for ch, wc in w_chans.items() if wc.value)
            if not _sel_ch:
                out_txt.value = _pre("Fit: select at least one channel (ion / p / n).")
                return
            sub = df_all_peaks[(df_all_peaks["peak_id"] == w_peak.value)
                               & (df_all_peaks["condition"].isin(w_conds.value))
                               & (df_all_peaks["T_nominal"].isin(w_temps.value))]
            try:
                res = fit_global_conductivity(sub, x=MODEL_EXPONENT, channels=_sel_ch)
            except ValueError as exc:
                out_txt.value = _pre(f"Fit: {exc}")
                return
            p, e = res["params"], res["perr"]
            parts = [f"Peak {w_peak.value}: R2={res['r2']:.4f}", f"n={res['n_points']}"]
            for ch in ("ion", "p", "n"):
                if ch not in _sel_ch:
                    continue
                if getattr(p, f"sigma0_{ch}") == 0.0 or np.isnan(e[f"Ea_{ch}"]):
                    parts.append(f"Ea_{ch}: not active (s0 = 0)")
                else:
                    parts.append(f"Ea_{ch}={getattr(p, f'Ea_{ch}'):.3f} eV")
            lines = ["  ".join(parts),
                     f"  points: {res['n_points']}  conditions: {len(w_conds.value)}"
                     f"  T: {sorted(int(t) for t in w_temps.value)}"]

            _po2_dir = RESULTS_BASE / "pO2"
            gtab = global_transference_table(sub, p, exponent=MODEL_EXPONENT)
            # Surface and residual map are rebuilt from the SAME subset the model
            # was just fitted on, so the scattered points always match the fit
            # (never the full dataset under a reduced model).
            _to_image(plot_brouwer_transference(sub, _po2_dir, sample_name=sample_id,
                                                peak_id=w_peak.value, exponent=MODEL_EXPONENT,
                                                df_t=gtab, params=p,
                                                perr=res["perr"]), img_bt)
            _to_image(plot_conductivity_surface_3d(sub, p, _po2_dir, sample_name=sample_id,
                                                   peak_id=w_peak.value), img_s)
            _to_image(plot_fit_residuals(sub, p, _po2_dir, sample_name=sample_id,
                                         peak_id=w_peak.value), img_r)
            lines.append(f"Figures overwritten in {_po2_dir.relative_to(NOTEBOOK_DIR)}: "
                         f"Brouwer_transference / Stage5_surface3D / Stage5_residuals "
                         f"Peak{w_peak.value}_{sample_id} (.png/.pdf)")

            if _btn is None:
                lines.append("(preview: parameters NOT saved; press ↻ Refit to save to session.json)")
                out_txt.value = _pre("\n".join(lines))
                return
            # Same schema as the batch cell, INCLUDING the Ea errors: stage5_params
            # is deep-merged in session.py, so omitting a key would leave the stale
            # value of the previous fit paired with the new parameters.
            if _update_session(stage5_params={str(w_peak.value): {
                "R2": res["r2"], "n_points": res["n_points"], "x": p.x,
                "sigma0_ion": p.sigma0_ion, "Ea_ion": p.Ea_ion, "Ea_ion_err": e["Ea_ion"],
                "sigma0_p": p.sigma0_p, "Ea_p": p.Ea_p, "Ea_p_err": e["Ea_p"],
                "sigma0_n": p.sigma0_n, "Ea_n": p.Ea_n, "Ea_n_err": e["Ea_n"]}}):
                lines.append(f"session.json updated (peak {w_peak.value}).")
            else:
                lines.append(_LOCKED_MSG)
            out_txt.value = _pre("\n".join(lines))
        except Exception:
            out_txt.value = _pre("Refit failed:\n" + _tb.format_exc())
        finally:
            _busy[0] = False
            w_go.disabled = False

    w_go.on_click(_on_refit)
    _display(W.VBox([W.HBox([w_peak, w_go]),
                     W.HBox([w_conds, w_temps,
                             W.VBox([W.Label("Channels:"), *w_chans.values()])]),
                     out_txt, img_bt, img_s, img_r]))
    _on_refit()
else:
    print("[INFO] interactive refit panel needs ipywidgets and a fitted dataset.")


## Output summary

`Results/pO2/`:
- `stage5_model.xlsx` - Parameters (6 params + `R2` + n per peak), Residuals, Metadata
- `Brouwer_transference_Peak*` - sigma vs `p(O2)` + t_ion, redrawn from the global model
- `Stage5_surface3D_Peak*` - fitted sigma(p(O2), T) surface with measured points
- `Stage5_residuals_Peak*` - relative-residual map over (`p(O2)`, T)

Each figure is saved as PNG (preview) and PDF (publication).